# Amazon Reviews — Sentiment Analysis

Loads Amazon product reviews, cleans text, and converts star ratings to binary
sentiment labels. Builds TF-IDF pipelines for Naive Bayes and Logistic Regression
classifiers, evaluating each on accuracy, precision, and recall.


In [31]:
import pandas as pd
import re
import string

# Load the dataset
df = pd.read_csv("C:/Users/mbuso/amazon_reviews.csv")

# Keep only relevant columns
df = df[['reviews.text', 'reviews.rating']].dropna()

# Convert ratings to binary sentiment: 1 (positive), 0 (negative/neutral)
df['sentiment'] = df['reviews.rating'].apply(lambda x: 1 if x >= 4 else 0)

# Basic cleaning function: lowercasing, removing punctuation and digits
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\d+", "", text)
    return text

# Apply text cleaning
df['clean_text'] = df['reviews.text'].apply(clean_text)
print("Data shape:", df.shape)
print("First few rows:")
print(df[['reviews.text', 'reviews.rating', 'sentiment', 'clean_text']].head())


Data shape: (1177, 4)
First few rows:
                                        reviews.text  reviews.rating  \
0  I initially had trouble deciding between the p...             5.0   
1  Allow me to preface this with a little history...             5.0   
2  I am enjoying it so far. Great for reading. Ha...             4.0   
3  I bought one of the first Paperwhites and have...             5.0   
4  I have to say upfront - I don't like coroporat...             5.0   

   sentiment                                         clean_text  
0          1  i initially had trouble deciding between the p...  
1          1  allow me to preface this with a little history...  
2          1  i am enjoying it so far great for reading had ...  
3          1  i bought one of the first paperwhites and have...  
4          1  i have to say upfront  i dont like coroporate ...  


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.pipeline import Pipeline

# Split data
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['sentiment'], test_size=0.2, random_state=42)

# Naive Bayes pipeline
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('nb', MultinomialNB())
])
nb_pipeline.fit(X_train, y_train)
nb_preds = nb_pipeline.predict(X_test)

# Logistic Regression pipeline
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('lr', LogisticRegression(max_iter=1000))
])
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

# Evaluation
print("Naive Bayes:")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print("Precision:", precision_score(y_test, nb_preds))
print("Recall:", recall_score(y_test, nb_preds))

print("\nLogistic Regression:")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print("Precision:", precision_score(y_test, lr_preds))
print("Recall:", recall_score(y_test, lr_preds))


Naive Bayes:
Accuracy: 0.8728813559322034
Precision: 0.8695652173913043
Recall: 1.0

Logistic Regression:
Accuracy: 0.8898305084745762
Precision: 0.8849557522123894
Recall: 1.0
